# 面试问题：为什么把 LLM Prefill 与 Decode 分离部署？KV Transfer、TTFT/TPOT 和路由怎样设计？

**一句话回答**：Prefill 对长 prompt 做高并行矩阵计算，通常更 compute-bound；Decode 每步只处理少量 token，却持续读取权重/KV，常更 memory-bound。混部会互相干扰并耦合并行配置。分离后可为两阶段独立扩容，但增加 KV 跨节点传输、排队、故障与一致性成本，路由应比较端到端 TTFT/TPOT 而非只看单阶段吞吐。

本 Notebook 手写工作负载合同、阶段成本、SLO、KV 传输、colocated/disaggregated 路由、容量配比、背压、幂等 transfer manifest 与 goodput，不依赖推理引擎。


In [ ]:
from dataclasses import dataclass
import hashlib,math
import numpy as np

# 参数只是教学用容量系数，不代表特定 GPU 实测。
SEED146=14601
assert SEED146==14601
assert math.ceil(9/4)==3
assert np.isfinite(np.array([1.])).all()


## 1. 请求合同同时包含 prompt、output 与两类 SLO

TTFT 从到达网关到首 token，包含队列、prefill、KV transfer 和调度；TPOT/TBT 描述后续 token 间隔。只按平均输入长度扩容会被长尾击穿，需保存长度预测区间、deadline、tenant 和取消语义。


In [ ]:
@dataclass(frozen=True)
class Request146:
    rid:str; prompt_tokens:int; max_output:int; ttft_slo_ms:float; tpot_slo_ms:float
    def __post_init__(self):
        # 非法长度或 SLO 在排队前直接拒绝。
        if self.prompt_tokens<=0 or self.max_output<=0 or min(self.ttft_slo_ms,self.tpot_slo_ms)<=0: raise ValueError("request_contract")
req146=Request146("r1",2048,256,500,40)
assert req146.prompt_tokens==2048
try: Request146("x",0,1,10,10); raise AssertionError("bad request")
except ValueError as e: assert str(e)=="request_contract"
assert req146.ttft_slo_ms>req146.tpot_slo_ms


## 2. 两阶段使用不同成本模型和并行策略

简化模型中 prefill 时间随输入 token 与 attention 成本增长，可用较大 batch/TP；decode 的单 token 时间受 batch、权重带宽和 KV 长度影响。估算器必须用 profile table 校准，并按模型、dtype、parallel plan 版本化。


In [ ]:
def phase_cost146(prompt,output,prefill_rate,decode_tpot):
    # rate 单位 token/ms；decode 返回每 token 时间与总占用。
    prefill_ms=prompt/prefill_rate; return {"prefill_ms":prefill_ms,"tpot_ms":decode_tpot,"decode_total_ms":output*decode_tpot}
cost146=phase_cost146(2048,256,20,12)
assert math.isclose(cost146["prefill_ms"],102.4)
assert cost146["decode_total_ms"]==3072
assert cost146["tpot_ms"]<req146.tpot_slo_ms


## 3. 分离的核心税是 KV Cache 传输

传输字节约为 `2·layers·tokens·Hkv·D·dtype_bytes`，还要加协议与分块元数据。传输时间由可用 RDMA/网络带宽和拥塞决定；长 prompt 的 KV 很大，若网络慢于 prefill 节省，分离会恶化 TTFT。


In [ ]:
def kv_transfer146(layers,tokens,hkv,d,bytes_,gbps):
    # Key/Value 两份，因此字节公式最前面乘 2。
    size=2*layers*tokens*hkv*d*bytes_; ms=size/(gbps*1e9)*1000; return size,ms
kv_size146,kv_ms146=kv_transfer146(32,2048,8,128,2,100)
assert kv_size146==268_435_456
assert math.isclose(kv_ms146,2.68435456)
assert kv_transfer146(32,4096,8,128,2,100)[0]==2*kv_size146


## 4. 路由比较端到端候选路径

colocated 路径可能有 prefill/decode interference；disaggregated 路径包含 P 队列、prefill、transfer 和 D 队列。路由先过滤 TPOT 不满足的 decode pool，再选预计 TTFT 最小且低于 deadline 的路径，预测版本写入 trace。


In [ ]:
def choose_route146(req,colocated,disaggregated):
    # 两条路径都用同一 TTFT/TPOT 定义，避免局部指标不可比。
    candidates=[]
    for name,x in [("colocated",colocated),("disaggregated",disaggregated)]:
        ttft=x["queue_p"]+x["prefill"]+x.get("transfer",0)+x["queue_d"]
        if x["tpot"]<=req.tpot_slo_ms: candidates.append((ttft,name))
    return min(candidates) if candidates else (math.inf,"reject")
coloc146={"queue_p":50,"prefill":130,"queue_d":0,"tpot":18}; dis146={"queue_p":20,"prefill":90,"transfer":kv_ms146,"queue_d":10,"tpot":12}
chosen146=choose_route146(req146,coloc146,dis146)
assert chosen146[1]=="disaggregated"
assert chosen146[0]<req146.ttft_slo_ms
assert choose_route146(req146,{**coloc146,"tpot":99},{**dis146,"tpot":99})[1]=="reject"


## 5. P/D 副本比由瓶颈到达率决定

每个 P 实例的 prompt-token/s 与每个 D 实例的 output-token/s 不同。给定流量长度分布，分别算两阶段总工作量和利用率，增加瓶颈侧副本；静态 1:1 在 prompt-heavy 与 generation-heavy 场景都可能浪费。


In [ ]:
def utilization146(rps,avg_prompt,avg_output,p_replicas,p_rate,d_replicas,d_rate):
    # 两阶段利用率分别计算，最大者决定系统 goodput 瓶颈。
    return rps*avg_prompt/(p_replicas*p_rate),rps*avg_output/(d_replicas*d_rate)
up146,ud146=utilization146(10,1000,100,2,8000,2,1000)
assert math.isclose(up146,.625)
assert math.isclose(ud146,.5)
assert max(utilization146(10,1000,500,2,8000,2,1000))>1


## 6. Backpressure 防止 P 生产 KV 速度超过 D 消费速度

D 队列或 transfer buffer 达高水位时，网关延迟/拒绝新 prefill，或降级到 colocated；否则大量已完成 KV 占用 HBM/网络并在 deadline 前过期。取消应传播到 P、transfer 和 D，并释放所有分块。


In [ ]:
def admission146(p_queue,d_queue,transfer_gb,limits):
    # 任一阶段越过高水位都停止继续放大下游压力。
    reasons=[]
    if p_queue>=limits["p"]: reasons.append("prefill_queue")
    if d_queue>=limits["d"]: reasons.append("decode_queue")
    if transfer_gb>=limits["gb"]: reasons.append("transfer_buffer")
    return (not reasons),reasons
ok146,reasons146=admission146(2,11,3,{"p":10,"d":10,"gb":5})
assert not ok146
assert reasons146==["decode_queue"]
assert admission146(1,1,1,{"p":10,"d":10,"gb":5})[0]


## 7. KV transfer 是带版本和幂等键的制品提交

manifest 绑定 request、model revision、layer/head layout、token range、dtype、RoPE/adapter revision、chunks 与 digest。D 端只有在所有 chunks 校验完成后才原子可见；重复发送同一 chunk 幂等，混入其他 revision 必须拒绝。


In [ ]:
def transfer_manifest146(rid,model,adapter,chunks):
    # canonical 字符串产生稳定 digest，供重试和接收端校验。
    body=f"{rid}|{model}|{adapter}|{','.join(map(str,chunks))}"; return {"rid":rid,"model":model,"adapter":adapter,"chunks":tuple(chunks),"digest":hashlib.sha256(body.encode()).hexdigest()}
m1_146=transfer_manifest146("r1","m7","a2",[0,1,2]); m2_146=transfer_manifest146("r1","m7","a2",[0,1,2])
assert m1_146==m2_146
assert len(m1_146["digest"])==64
assert transfer_manifest146("r1","m8","a2",[0,1,2])["digest"]!=m1_146["digest"]


## 8. 用同时满足 TTFT 与 TPOT 的 goodput 验收

吞吐高但大量请求违反任一 SLO 不算可用容量。按 prompt/output 长度、tenant、cache hit、故障 slice 报 goodput、p50/p95 TTFT/TPOT、transfer bytes/time、队列与利用率；与 colocated 做相同硬件成本对照和峰值突发测试。


In [ ]:
samples146=[{"ttft":200,"tpot":20},{"ttft":600,"tpot":20},{"ttft":300,"tpot":50},{"ttft":450,"tpot":30}]
# goodput 只计两个 SLO 同时达标的请求。
passed146=[x for x in samples146 if x["ttft"]<=req146.ttft_slo_ms and x["tpot"]<=req146.tpot_slo_ms]
assert len(passed146)==2
assert len(passed146)/len(samples146)==.5
assert max(x["ttft"] for x in passed146)<=500


## 面试总结

完整回答是：**区分 compute-bound prefill 与 bandwidth-bound decode → 同时定义 TTFT/TPOT → 独立 profile/parallel plan → 计算 KV transfer 字节与拥塞 → 比较 colocated/disaggregated 端到端路径 → 按长度分布配 P/D 副本 → 高水位背压/取消 → manifest 幂等传输与原子可见 → 用双 SLO goodput 和成本验收**。P/D 分离是资源解耦，不是无代价加速。

延伸阅读：[DistServe](https://arxiv.org/abs/2401.09670)、[Splitwise](https://arxiv.org/abs/2311.18677)、[Mooncake](https://arxiv.org/abs/2407.00079)。
